In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!uv pip install transformers timm --quiet

In [3]:
import warnings

warnings.filterwarnings("ignore", module="torchaudio")
warnings.filterwarnings("ignore")

In [4]:
import os
# from collections import defaultdict
import pandas as pd
import itertools
import librosa
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
import soundfile as sf
from pathlib import Path
from collections import defaultdict
from joblib import Parallel, delayed


import random
import torch
import torch.nn as nn
import pytorch_lightning as pl
import torchmetrics
import wandb
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, random_split
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint
import torchaudio.transforms as T

from transformers import ASTForAudioClassification
from transformers.optimization import get_cosine_schedule_with_warmup

2026-03-30 14:46:47.883737: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774882008.087746      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774882008.144675      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774882008.643689      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774882008.643734      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774882008.643737      55 computation_placer.cc:177] computation placer alr

In [5]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
WANDB_API_KEY = user_secrets.get_secret("wandb_api_key")

In [6]:
# !wandb login

In [7]:

try:
    wandb.login(key=WANDB_API_KEY)
except:
    wandb.login()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Currently logged in as: 23f2000873 (23f2000873-dl-genai-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# EDA

## Class Distribution

In [8]:
# training_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

# genre_counts = {}

# for genre in os.listdir(training_path):
#     genre_path = os.path.join(training_path, genre)
#     if os.path.isdir(genre_path):
#         songs = os.listdir(genre_path)
#         genre_counts[genre] = len(songs)

# df_genres = pd.DataFrame(list(genre_counts.items()), columns=["Genre", "Num_Songs"])
# df_genres.sort_values("Num_Songs", ascending=False)

Class distribution is balanced

## Audio Length Distribution

In [9]:
# durations = []

# for genre in os.listdir(training_path):
#     genre_path = os.path.join(training_path, genre)
    
#     for song in tqdm(os.listdir(genre_path)):
#         song_path = os.path.join(genre_path, song, "vocals.wav")  # pick 1 stem
#         y, sr = librosa.load(song_path, sr=None)
#         durations.append(len(y)/sr)

# pd.Series(durations).describe()

In [10]:
# pd.Series(durations).value_counts(bins=10).sort_index()

In [11]:

# plt.hist(durations, bins=20)
# plt.title("Audio Length Distribution")
# plt.xlabel("Seconds")
# plt.ylabel("Count")
# plt.show()

# Mashup Train dataset creation

## Stage 1

In [12]:

class MashupCreatorV1:
    """
    Generator that strictly follows competition description.

    Steps:
    1) Pick 4 stems from different songs of same genre
    2) Tempo-align them
    3) Linear mix
    4) Add ESC-50 noise at random positions
    5) RMS normalize
    """

    def __init__(self,
                 stems_root,
                 esc50_root,
                 output_dir="/kaggle/working/mashups_v4",
                 target_sr=22500,
                 target_duration=30.0):

        self.stems_root = Path(stems_root)
        self.esc50_root = Path(esc50_root)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)

        self.target_sr = target_sr
        self.target_samples = int(target_sr * target_duration)
        self.audio_cache = {}
        self.genre_map = self._index_dataset()
        self.noise_files = list(self.esc50_root.rglob("*.wav"))

    def _index_dataset(self):
        genre_map = defaultdict(dict)

        for genre_dir in self.stems_root.iterdir():
            if not genre_dir.is_dir():
                continue

            for song_dir in genre_dir.iterdir():
                stems = list(song_dir.glob("*.wav"))
                if len(stems) == 4:
                    genre_map[genre_dir.name][song_dir.name] = stems

        return genre_map

    def load_audio(self, path):
        p=str(path)
        if p in self.audio_cache:
            return self.audio_cache[p]

        y,sr=librosa.load(path, sr=None, mono=True) # fix set sr = self.target_sr or duration=30s, 
        if sr!=self.target_sr:
            y=librosa.resample(y, orig_sr=sr, target_sr=self.target_sr)

        y=y.astype(np.float32) # default is fload32
        self.audio_cache[p]=y
        return y

    def fix_length(self, y):
        if len(y) > self.target_samples:
            return y[:self.target_samples]
        return np.pad(y, (0, self.target_samples - len(y)))


    # def linear_mix(self, stems):
    #     mix = np.sum(stems, axis=0)
    #     return mix

    def add_noise_segments(self, waveform):
    
        if not self.noise_files:
            return waveform
    
        waveform = waveform.copy()
    
        num_noises = random.randint(1, 2)
    
        for _ in range(num_noises):
            noise_path = random.choice(self.noise_files)
            noise = self.load_audio(noise_path)
    
            segment_sec = random.uniform(3, 8)
            segment_samples = int(segment_sec * self.target_sr)
    
            segment_samples = min(segment_samples, self.target_samples)
    
            if len(noise) < segment_samples:
                noise = np.pad(noise, (0, segment_samples - len(noise)))
            else:
                start = random.randint(0, len(noise) - segment_samples)
                noise = noise[start:start + segment_samples]
    
            insert_pos = random.randint(
                0, self.target_samples - segment_samples
            )
    
            scale = random.uniform(0.05, 0.3)
    
            waveform[insert_pos:insert_pos + segment_samples] += noise * scale
    
        return waveform


    def rms_normalize(self, y, target_rms=0.1):
        rms = np.sqrt(np.mean(y ** 2))
        return y * (target_rms / (rms + 1e-8))

    def create_mashup(self, mashup_id):
        genre = random.choice(list(self.genre_map.keys()))
        songs = random.sample(list(self.genre_map[genre].keys()), 4)
        
        stem_types = itertools.cycle(["vocals.wav", "drums.wav", "bass.wav", "other.wav"])
    
        waveforms = []
    
        for song, stem_name in zip(songs, stem_types):
            stem_path = self.stems_root / genre / song / stem_name
            y = self.load_audio(stem_path)
            y = self.fix_length(y)
            waveforms.append(y)
            
    
        mashup = np.sum(waveforms, axis=0) # why not use the function
    
        mashup = self.add_noise_segments(mashup)
        mashup = self.rms_normalize(mashup)
    
        output_path = self.output_dir / f"{mashup_id}_{genre}.wav"
    
        sf.write(str(output_path), mashup, self.target_sr)
    
        return str(output_path)


    def generate_batch(self, n_mashups, n_jobs=-1):

        Parallel(n_jobs=n_jobs, prefer="threads")(
            delayed(self.create_mashup)(f"mashup_{i:05d}")
            for i in tqdm(range(n_mashups))
        )

In [13]:
creator1 = MashupCreatorV1(
    stems_root="/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems",
    esc50_root="/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master"
)

# creator1.generate_batch(
#     n_mashups=2500, # number of mashups to generate
#     n_jobs=-1 
# )

## stage 2

In [14]:

def fast_time_stretch(y, rate):
    if abs(rate - 1.0) < 0.02:
        return y
    n = int(len(y) / rate)
    return np.interp(
        np.linspace(0, len(y), n),
        np.arange(len(y)),
        y
    ).astype(np.float32)

def fix_length(y, target):
    if len(y) > target:
        start = random.randint(0, len(y)-target)
        return y[start:start+target]
    return np.pad(y, (0, target-len(y)))

In [15]:

class MashupCreatorV2:

    def __init__(self, stems_root, esc50_root,
                 output_dir="/kaggle/working/mashups_v5",
                 target_sr=22050, target_duration=30):

        self.stems_root = Path(stems_root)
        self.esc50_root = Path(esc50_root)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)

        self.target_sr = target_sr
        self.target_samples = target_sr * target_duration

        self.genre_map = self._index_dataset()
        self.noise_files = list(self.esc50_root.rglob("*.wav"))

        self.audio_cache = creator1.audio_cache or {}
        self.tempo_cache = {}
        self.cache_path = self.output_dir/"tempo_cache.npy"

        if self.cache_path.exists():
            self.tempo_cache = np.load(self.cache_path, allow_pickle=True).item()

    def _index_dataset(self):
        from collections import defaultdict
        genre_map = defaultdict(dict)
        for g in self.stems_root.iterdir():
            if g.is_dir():
                for s in g.iterdir():
                    stems = list(s.glob("*.wav"))
                    if len(stems)==4:
                        genre_map[g.name][s.name]=stems
        return genre_map

    def load_audio(self, path):
        p=str(path)
        if p in self.audio_cache:
            return self.audio_cache[p]

        y,sr=librosa.load(path, sr=None, mono=True)
        if sr!=self.target_sr:
            y=librosa.resample(y, orig_sr=sr, target_sr=self.target_sr)

        y=y.astype(np.float32)
        self.audio_cache[p]=y
        return y

    def get_tempo(self, path):
    
        p = str(path)
    
        if p in self.tempo_cache:
            return self.tempo_cache[p]
    
        y = self.load_audio(path)
        tempo, _ = librosa.beat.beat_track(y=y, sr=self.target_sr)
    
        if isinstance(tempo, np.ndarray):
            tempo = tempo[0] if tempo.size > 0 else 120
    
        tempo = float(tempo if tempo > 0 else 120)
    
        self.tempo_cache[p] = tempo
    
        return tempo

    def tempo_align(self, stems, paths):

        tempos=[self.get_tempo(p) for p in paths]
        ref=np.median(tempos)

        out=[]
        for y,t in zip(stems,tempos):
            rate=np.clip(t/ref,0.8,1.2)
            y=librosa.effects.time_stretch(y, rate=rate)
            y=fix_length(y,self.target_samples)
            out.append(y)

        return out

    def add_noise(self, mix):

        sig_pow=np.mean(mix**2)+1e-8
        n_noises=np.random.poisson(1.5)+1

        for _ in range(n_noises):
            noise=self.load_audio(random.choice(self.noise_files))

            seg=int(random.uniform(2,5)*self.target_sr)
            noise=fix_length(noise,seg)

            pos=random.randint(0,self.target_samples-seg)

            snr=random.uniform(-5,15)
            noise_pow=np.mean(noise**2)+1e-8
            scale=np.sqrt(sig_pow/(10**(snr/10)*noise_pow))

            mix[pos:pos+seg]+=noise*scale

        return mix

    def create_mashup(self, idx):

        genre=random.choice(list(self.genre_map.keys()))
        songs=random.sample(list(self.genre_map[genre].keys()),4)

        stems_names=["vocals.wav","drums.wav","bass.wav","other.wav"]

        ys=[]
        paths=[]

        for song,stem in zip(songs,stems_names):
            p=self.stems_root/genre/song/stem
            y=self.load_audio(p)
            y=fix_length(y,self.target_samples)
            ys.append(y)
            paths.append(p)

        ys=self.tempo_align(ys,paths)

        # gain randomization
        ys=[y*random.uniform(0.5,1.3) for y in ys]

        mix=np.sum(ys,axis=0)
        mix=self.add_noise(mix)

        mix/=np.max(np.abs(mix)+1e-8) # peak normalization
        mix*=0.95

        import soundfile as sf
        sf.write(self.output_dir/f"{idx:05d}_{genre}.wav",mix,self.target_sr)

    def generate(self,n=3000,n_jobs=-1):

        Parallel(n_jobs=n_jobs,prefer="threads")(
            delayed(self.create_mashup)(i)
            for i in tqdm(range(n))
        )

In [16]:
creator2 = MashupCreatorV2(
    stems_root="/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems",
    esc50_root="/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master"
)

# creator2.generate(
#     n=7500, # number of mashups to generate
#     n_jobs=-1 
# )

In [17]:
# np.save(creator2.cache_path, creator2.tempo_cache)

## stage 3

In [18]:

def fast_time_stretch(y, rate):
    if abs(rate - 1.0) < 0.02:
        return y
    n = int(len(y) / rate)
    return np.interp(
        np.linspace(0, len(y), n),
        np.arange(len(y)),
        y
    ).astype(np.float32)
# use librosa.effects.time_stretch for next time

def fix_length(y, target):
    if len(y) > target:
        start = random.randint(0, len(y)-target)
        return y[start:start+target]
    return np.pad(y, (0, target-len(y)))


class MashupCreatorV3:

    def __init__(self,
                 stems_root,
                 esc50_root,
                 output_dir="/kaggle/working/mashups_adv",
                 target_sr=22050,
                 target_duration=30):

        self.stems_root = Path(stems_root)
        self.esc50_root = Path(esc50_root)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)

        self.target_sr = target_sr
        self.target_samples = target_sr * target_duration

        self.genre_map = self._index_dataset()
        self.noise_files = list(self.esc50_root.rglob("*.wav"))

        self.audio_cache = creator1.audio_cache or {}

    def _index_dataset(self):
        genre_map = defaultdict(dict)

        for genre_dir in self.stems_root.iterdir():
            if genre_dir.is_dir():
                for song_dir in genre_dir.iterdir():
                    stems = list(song_dir.glob("*.wav"))
                    if len(stems) == 4:
                        genre_map[genre_dir.name][song_dir.name] = stems

        return genre_map

    def load_audio(self, path):
        p = str(path)

        if p in self.audio_cache:
            return self.audio_cache[p]

        y, sr = librosa.load(path, sr=None, mono=True)

        if sr != self.target_sr:
            y = librosa.resample(y, orig_sr=sr, target_sr=self.target_sr)

        y = y.astype(np.float32)
        self.audio_cache[p] = y
        return y

    def add_noise(self, mix):

        sig_pow = np.mean(mix**2) + 1e-8
        n_noises = random.randint(2, 4)

        for _ in range(n_noises):

            noise = self.load_audio(random.choice(self.noise_files))

            seg_len = int(random.uniform(1, 5) * self.target_sr)
            noise = fix_length(noise, seg_len)

            pos = random.randint(0, self.target_samples - seg_len)

            snr = random.uniform(-10, 5)

            noise_pow = np.mean(noise**2) + 1e-8
            scale = np.sqrt(sig_pow/(10**(snr/10)*noise_pow))

            mix[pos:pos+seg_len] += noise * scale

        return mix

    def create_mashup(self, idx):

        genre = random.choice(list(self.genre_map.keys()))
        songs = random.sample(list(self.genre_map[genre].keys()), 4)

        stems = ["vocals.wav","drums.wav","bass.wav","other.wav"]

        waveforms = []

        for song, stem in zip(songs, stems):

            y = self.load_audio(self.stems_root/genre/song/stem)

            # stem dropout
            if random.random() < 0.2:
                y = np.zeros_like(y)

            rate = random.uniform(0.75, 1.25)
            y = librosa.effects.time_stretch(y, rate=rate)

            y = fix_length(y, self.target_samples)

            y *= random.uniform(0.3, 1.7) # gain

            waveforms.append(y)

        mix = np.sum(waveforms, axis=0)

        mix = self.add_noise(mix)

        # peak normalize
        peak = np.max(np.abs(mix)) + 1e-8
        mix = mix / peak * 0.95

        sf.write(
            self.output_dir/f"adv_{idx:05d}_{genre}.wav",
            mix,
            self.target_sr
        )

    def generate(self, n=2000, n_jobs=-1):

        Parallel(n_jobs=n_jobs, prefer="threads")(
            delayed(self.create_mashup)(i)
            for i in tqdm(range(n))
        )

In [19]:
creator_adv = MashupCreatorV3(
    stems_root="/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems",
    esc50_root="/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master"
)

# creator_adv.generate(n=3500)

# CRNN

In [20]:
GENRES = [
    "blues",
    "classical",
    "country",
    "disco",
    "hiphop",
    "jazz",
    "metal",
    "pop",
    "reggae",
    "rock"
]

GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for g, i in GENRE_TO_IDX.items()}

In [21]:
class MashupDataset(Dataset):
    def __init__(self, root_dir, sr=22500, n_mels=128, duration=30,
                 return_filenames=False):

        self.files = sorted(list(Path(root_dir).glob("*.wav")))
        self.sr = sr
        self.n_mels = n_mels
        self.samples = sr * duration
        self.return_filenames = return_filenames

    def __len__(self):
        return len(self.files)

    def load_mel(self, path):
        y, _ = librosa.load(path, sr=self.sr)

        if len(y) < self.samples:
            y = np.pad(y, (0, self.samples - len(y)))
        else:
            y = y[:self.samples]

        mel = librosa.feature.melspectrogram(
            y=y,
            sr=self.sr,
            n_fft=1024,
            hop_length=512,
            n_mels=self.n_mels
        )

        mel = librosa.power_to_db(mel).astype(np.float32)
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)

        return torch.tensor(mel).unsqueeze(0)

    def __getitem__(self, idx):
        file = self.files[idx]
        x = self.load_mel(file)

        if self.return_filenames:
            return x, file.name

        genre = file.stem.split("_")[-1]
        y = torch.tensor(GENRE_TO_IDX[genre])

        return x, y

In [22]:
class MashupDataModule(pl.LightningDataModule):
    def __init__(self,
                 train_dir=None,
                 test_dir=None,
                 batch_size=16):

        super().__init__()
        self.train_dir = train_dir
        self.test_dir = test_dir
        self.batch_size = batch_size

    def setup(self, stage=None):

        if self.train_dir is not None:
            full_dataset = MashupDataset(self.train_dir)

            val_size = int(0.2 * len(full_dataset))
            train_size = len(full_dataset) - val_size

            self.train_ds, self.val_ds = random_split(
                full_dataset, [train_size, val_size]
            )

        if self.test_dir is not None:
            self.test_ds = MashupDataset(
                self.test_dir,
                return_filenames=True
            )

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=4,
            pin_memory=True
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )

    def predict_dataloader(self):
        return DataLoader(
            self.test_ds,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )

In [23]:


class CRNNLightning(pl.LightningModule):
    def __init__(self, n_classes=10, lr=3e-4):
        super().__init__()
        self.save_hyperparameters()

        # CNN
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.gru = nn.GRU(
            input_size=128 * 16,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        self.attn = nn.Linear(512, 1)

        self.fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, n_classes)
        )

        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

        self.val_f1 = torchmetrics.F1Score(
            task="multiclass",
            num_classes=n_classes,
            average="macro"
        )

    def forward(self, x):
        x = self.cnn(x)

        B, C, F, T = x.shape
        x = x.permute(0, 3, 1, 2).reshape(B, T, C * F)

        x, _ = self.gru(x)

        attn_weights = torch.softmax(self.attn(x), dim=1)
        x = torch.sum(x * attn_weights, dim=1)

        return self.fc(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)

        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)

        loss = self.loss_fn(logits, y)
        preds = torch.argmax(logits, dim=1)

        self.val_f1(preds, y)

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_macro_f1", self.val_f1, prog_bar=True)
        
    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        x, filenames = batch
    
        logits = self(x)
        preds = torch.argmax(logits, dim=1)
    
        return preds, filenames


    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=1e-4
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=20
        )

        return [optimizer], [scheduler]

In [24]:
# import wandb
# from pytorch_lightning.loggers import WandbLogger
# from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint


# wandb_logger = WandbLogger(project="mashup-crnn-v3")

# early_stop = EarlyStopping(
#     monitor="val_macro_f1",
#     mode="max",
#     patience=5
# )

# checkpoint = ModelCheckpoint(
#     monitor="val_macro_f1",
#     mode="max",
#     save_top_k=1
# )

## training CRNN

In [25]:
# from pytorch_lightning import Trainer

# data_module = MashupDataModule(
#     train_dir="/kaggle/working/mashups_v4",
#     batch_size=16
# )

# model = CRNNLightning(n_classes=10)

# trainer = Trainer(
#     max_epochs=30,
#     accelerator="auto",
#     devices="auto",
#     precision=16,
#     logger=wandb_logger,
#     callbacks=[early_stop, checkpoint],
#     gradient_clip_val=1.0
# )

# trainer.fit(model, data_module)

NameError: name 'wandb_logger' is not defined

In [ ]:
# from pathlib import Path
# import kagglehub

# model_path_crnn = Path(checkpoint.best_model_path).parent

# KAGGLE_USERNAME = "a6du101"
# FRAMEWORK = "pytorch"
# VARIATION = "v2"
# MODEL = "mashup-crnn"

# # Kaggle Hub handle
# handle_crnn = f"{KAGGLE_USERNAME}/{MODEL}/{FRAMEWORK}/{VARIATION}"


# kagglehub.model_upload(
#     handle_crnn,
#     model_path_crnn,
#     version_notes="CRNN with 7430 updated mashups"
# )

# print(f"Uploaded CRNN model to {handle_crnn}")

# inference Crnn

In [ ]:

# model_dir = "/kaggle/input/models/a6du101/mashup-crnn/pytorch/v2/1"
# ckpt_path = list(Path(model_dir).rglob("*.ckpt"))[0]

# print("Checkpoint:", ckpt_path)

In [ ]:
# model = CRNNLightning.load_from_checkpoint(
#     checkpoint_path=ckpt_path,
#     n_classes=10
# )

# model.eval()

In [ ]:
# data_module = MashupDataModule(
#     test_dir="/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups",
#     batch_size=16
# )

# trainer = pl.Trainer(
#     accelerator="auto",
#     devices="auto",
#     precision=16,
#     logger=False
# )

# predictions = trainer.predict(model, datamodule=data_module)

In [ ]:
# all_preds = []
# all_filenames = []

# for batch_preds, batch_files in predictions:
#     all_preds.extend(batch_preds.cpu().numpy())
#     all_filenames.extend(batch_files)

# predicted_genres = [IDX_TO_GENRE[p] for p in all_preds]


# test_df = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv")

# print(len(predicted_genres), len(test_df))

# test_df["genre"] = predicted_genres

# submission = test_df[["id", "genre"]]
# submission.to_csv("submission.csv", index=False)

# submission.head()

# AST

In [26]:
from transformers import ASTFeatureExtractor

feature_extractor = ASTFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [81]:
class ASTMashupDataset(Dataset):
    def __init__(self, root_dir, train=True, return_filenames=False):

        self.files = sorted(list(Path(root_dir).glob("*.flac")))
        self.train = train
        self.return_filenames = return_filenames

        self.feature_extractor = ASTFeatureExtractor.from_pretrained(
            "MIT/ast-finetuned-audioset-10-10-0.4593"
        )

        self.target_sr = 16000
        self.target_duration = 30
        self.target_samples = self.target_sr * self.target_duration

    def __len__(self):
        return len(self.files)

    # def __getitem__(self, idx):
    #     file = self.files[idx]

    #     waveform, sr = librosa.load(file, sr=self.target_sr)
        
    #     # Random crop 10 sec during training
    #     if len(waveform) > self.target_samples:
    #         if self.train:
    #             start = np.random.randint(0, len(waveform) - self.target_samples)
    #         else:
    #             start = (len(waveform) - self.target_samples) // 2 #28feb
    #             # start = 0
    #         waveform = waveform[start:start + self.target_samples]
    #     else:
    #         waveform = np.pad(waveform, (0, self.target_samples - len(waveform)))

    #     # official feature extractor
    #     inputs = self.feature_extractor(
    #         waveform,
    #         sampling_rate=self.target_sr,
    #         return_tensors="pt"
    #     )

    #     x = inputs["input_values"].squeeze(0)

    #     if self.return_filenames:
    #         return x, file.name

    #     genre = file.stem.split("_")[-1]
    #     y = torch.tensor(GENRE_TO_IDX[genre])

    #     return x, y
    def __getitem__(self, idx):
        file = self.files[idx]
    
        waveform, sr = librosa.load(file, sr=self.target_sr)
    
        # === SPLIT INTO MULTIPLE 10s CHUNKS ===
        chunks = []
    
        total_len = len(waveform)
    
        if total_len >= self.target_samples:
            n_chunks = total_len // self.target_samples
    
            for i in range(n_chunks):
                start = i * self.target_samples
                chunk = waveform[start:start + self.target_samples]
                chunks.append(chunk)
    
        else:
            padded = np.pad(waveform, (0, self.target_samples - total_len))
            chunks.append(padded)
    
        # === Convert each chunk to AST input ===
        inputs = self.feature_extractor(
            chunks,
            sampling_rate=self.target_sr,
            return_tensors="pt"
        )
    
        x = inputs["input_values"]  # [N, T, F]
    
        genre = file.stem.split("_")[-1]
        y = torch.tensor(GENRE_TO_IDX[genre])
    
        return x, y



In [82]:

class ASTMashupDataModule(pl.LightningDataModule):
    def __init__(self,
                 train_dir=None,
                 test_dir=None,
                 batch_size=8):

        super().__init__()
        self.train_dir = train_dir
        self.test_dir = test_dir
        self.batch_size = batch_size

    def setup(self, stage=None):

        if self.train_dir is not None:
            full_dataset = ASTMashupDataset(
                self.train_dir,
                train=True
            )

            val_size = int(0.2 * len(full_dataset))
            train_size = len(full_dataset) - val_size

            self.train_ds, self.val_ds = random_split(
                full_dataset,
                [train_size, val_size]
            )

            self.val_ds.dataset.train = False

        if self.test_dir is not None:
            self.test_ds = ASTMashupDataset(
                self.test_dir,
                train=False,
                return_filenames=True
            )

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=4,
            pin_memory=True
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )

    def predict_dataloader(self):
        return DataLoader(
            self.test_ds,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )

## with predict step

In [83]:

class ASTLightning(pl.LightningModule):
    def __init__(self, n_classes=10, lr=2e-5, weight_decay=0.05,
                 warmup_ratio=0.1, freeze_backbone_epochs=3):

        super().__init__()
        self.save_hyperparameters()

        self.model = ASTForAudioClassification.from_pretrained(
            "MIT/ast-finetuned-audioset-10-10-0.4593",
            num_labels=n_classes,
            ignore_mismatched_sizes=True
        )

        self.chunk_attention = nn.Sequential(
            nn.Linear(n_classes, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

        self.val_f1 = torchmetrics.F1Score(
            task="multiclass",
            num_classes=n_classes,
            average="macro"
        )

        self.freeze_backbone_epochs = freeze_backbone_epochs
        self.backbone_frozen = False

    def forward(self, x):
        return self.model(x).logits

    # def training_step(self, batch, batch_idx):
    #     x, y = batch
    #     logits = self(x)
    #     loss = self.loss_fn(logits, y)

    #     self.log("train_loss", loss, prog_bar=True)
    #     self.log("lr", self.trainer.optimizers[0].param_groups[0]["lr"], prog_bar=True)
    #     return loss
    def training_step(self, batch, batch_idx):
        x, y = batch  # [B, N, T, F]
        B, N, T, F = x.shape
    
        x = x.view(B * N, T, F)
    
        logits = self(x)  # [B*N, C]
    
        # reshape
        logits = logits.view(B, N, -1)  # [B, N, C]
    
        # === ATTENTION ===
        attn_scores = self.chunk_attention(logits)  # [B, N, 1]
        attn_weights = torch.softmax(attn_scores, dim=1)
    
        final_logits = torch.sum(attn_weights * logits, dim=1)  # [B, C]
    
        loss = self.loss_fn(final_logits, y)
    
        self.log("train_loss", loss, prog_bar=True)
        return loss

    # def validation_step(self, batch, batch_idx):
    #     x, y = batch
    #     logits = self(x)
    #     loss = self.loss_fn(logits, y)

    #     preds = torch.argmax(logits, dim=1)
    #     self.val_f1(preds, y)

    #     self.log("val_loss", loss, prog_bar=True)
    #     self.log("val_macro_f1", self.val_f1, prog_bar=True)
    def validation_step(self, batch, batch_idx):
        x, y = batch
        B, N, T, F = x.shape
    
        x = x.view(B * N, T, F)
    
        logits = self(x)
        logits = logits.view(B, N, -1)
    
        # === ATTENTION ===
        attn_scores = self.chunk_attention(logits)
        attn_weights = torch.softmax(attn_scores, dim=1)
    
        final_logits = torch.sum(attn_weights * logits, dim=1)
    
        loss = self.loss_fn(final_logits, y)
    
        preds = torch.argmax(final_logits, dim=1)
    
        self.val_f1(preds, y)
    
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_macro_f1", self.val_f1, prog_bar=True)

    def on_train_epoch_start(self):
        if self.current_epoch < self.freeze_backbone_epochs:
            if not self.backbone_frozen:
                for param in self.model.audio_spectrogram_transformer.parameters():
                    param.requires_grad = False
                self.backbone_frozen = True
        else:
            if self.backbone_frozen:
                for param in self.model.audio_spectrogram_transformer.parameters():
                    param.requires_grad = True
                self.backbone_frozen = False

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        # crops: [Batch, 5, 1024, 128]
        crops, filenames = batch
        B, N, T, F = crops.shape

        # Flatten Batch and Crops 
        crops = crops.view(B * N, T, F)
        
        logits = self(crops) # [B*N, 10]
        

        probs = torch.softmax(logits, dim=-1) # [B*N, 10]
        
        probs = probs.view(B, N, -1)
        

        mean_probs = torch.mean(probs, dim=1)
        
        max_probs, _ = torch.max(probs, dim=1)

        final_probs = (0.7 * mean_probs) + (0.3 * max_probs)
        
        preds = torch.argmax(final_probs, dim=1)
        
        return {
            "preds": preds,
            "filenames": filenames
        }
    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        crops, filenames = batch
        B, N, T, F = crops.shape
    
        crops = crops.view(B * N, T, F)
    
        logits = self(crops)
        logits = logits.view(B, N, -1)
    
        # === ATTENTION ===
        attn_scores = self.chunk_attention(logits)
        attn_weights = torch.softmax(attn_scores, dim=1)
    
        final_logits = torch.sum(attn_weights * logits, dim=1)
    
        preds = torch.argmax(final_logits, dim=1)
    
        return {
            "preds": preds,
            "filenames": filenames
        }

    def configure_optimizers(self):

        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay
        )

        total_steps = self.trainer.estimated_stepping_batches
        warmup_steps = int(self.hparams.warmup_ratio * total_steps)

        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )
        

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step"
            }
        }

In [61]:
# wandb.init(project="mashup-ast")

# wandb_logger_ast = WandbLogger(
#     project="mashup-ast",
#     log_model=False
# )

from pytorch_lightning.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    LearningRateMonitor
)

early_stop = EarlyStopping(
    monitor="val_macro_f1",
    mode="max",
    patience=4
)

checkpoint = ModelCheckpoint(
    monitor="val_macro_f1",
    mode="max",
    save_top_k=1,
    filename="best-ast-{epoch}-{val_macro_f1:.4f}"
)

In [62]:
# ast_data_module = ASTMashupDataModule(
#     train_dir="/kaggle/working/mashups_v4",
#     batch_size=8
# )


# model = ASTLightning(
#     n_classes=10,
#     lr=2e-5,
#     weight_decay=0.05,
#     warmup_ratio=0.1,
#     freeze_backbone_epochs=3
# )


# trainer = pl.Trainer(
#     max_epochs=16,
#     accelerator="auto",
#     devices=1,
#     precision='16-mixed',
#     logger=wandb_logger_ast,
#     gradient_clip_val=1.0,
#     log_every_n_steps=10
# )

# Traninng AST

In [63]:
# trainer.fit(model, ast_data_module)

## uploading AST

In [64]:
# Path where Lightning saved the best checkpoint
# model_path_crnn = Path(checkpoint.best_model_path).parent


KAGGLE_USERNAME = "a6du101"
FRAMEWORK = "pytorch"
VARIATION = "v1"
MODEL = "mashup-ast-cpkt"

# Kaggle Hub handle
handle_crnn = f"{KAGGLE_USERNAME}/{MODEL}/{FRAMEWORK}/{VARIATION}"


# kagglehub.model_upload(
#     handle_crnn,
#     model_path_crnn,
#     version_notes="AST with 7430 updated mashups"
# )

print(f"Uploaded CRNN model to {handle_crnn}")

Uploaded CRNN model to a6du101/mashup-ast-cpkt/pytorch/v1


# 3 stage ast traning

## 1st Stage training

In [65]:
v4 = Path('/kaggle/input/notebooks/a6du101/masups/mashups_v4')
num_files = len([p for p in v4.iterdir() if p.is_file()])
print(num_files)

3000


In [77]:
# # wandb.init(project="mashup-ast-3stage")

wandb_logger_ast = WandbLogger(
    project="mashup-ast-final",
    log_model=False
)
ast_data_module_stage1 = ASTMashupDataModule(
    train_dir="/kaggle/input/notebooks/a6du101/masups/mashups_v4",
    batch_size=12
)

model_stage1 = ASTLightning(
    n_classes=10,
    lr=1e-5,
    weight_decay=0.07,
    warmup_ratio=0.12,
    freeze_backbone_epochs=2
)

trainer1 = pl.Trainer(
    max_epochs=16,
    accelerator="auto",
    devices=1,
    precision="16-mixed",
    logger=wandb_logger_ast,
    callbacks=[early_stop, checkpoint],
    gradient_clip_val=1.0,
    log_every_n_steps=10
)

trainer1.fit(model_stage1, datamodule=ast_data_module_stage1)

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model           │ ASTForAudioClassification │ 86.2 M │ eval  │     0 │
│ 1 │ chunk_attention │ Sequential                │  1.5 K │ train │     0 │
│ 2 │ loss_fn         │ CrossEntropyLoss          │      0 │ train │     0 │
│ 3 │ val_f1          │ MulticlassF1Score         │      0 │ train │     0 │
└───┴─────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 86.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 86.2 M                                                                                               
Total estimated model params size (MB): 344                                                                        
Modules in train mode: 6                                                                                           
Modules in eval mode: 216                                                                                          
Total FLOPs: 0

Output()

In [71]:
# stage1_ckpt = "/kaggle/input/models/a6du101/ast-step1-trained/pytorch/v1/2/best-ast-stage2-epoch5-val_macro_f10.9765.ckpt"
stage1_ckpt=checkpoint.best_model_path

In [78]:
stage1_ckpt

'./mashup-ast-final/mn85tmuh/checkpoints/best-ast-epoch=4-val_macro_f1=0.9737.ckpt'

## 2nd stage training

In [80]:

wandb_logger_ast2 = WandbLogger(
    project="mashup-ast-final",
    log_model=False,
)
early_stop_stage2 = EarlyStopping(
    monitor="val_macro_f1",
    mode="max",
    patience=3
)

checkpoint_stage2 = ModelCheckpoint(
    monitor="val_macro_f1",
    mode="max",
    save_top_k=1,
    filename="best-ast-stage2-{epoch}-{val_macro_f1:.4f}"
)

ast_data_module_stage2 = ASTMashupDataModule(
    train_dir="/kaggle/input/notebooks/a6du101/masups/mashups_v5",
    batch_size=8
)
model_stage2 = ASTLightning.load_from_checkpoint(
    stage1_ckpt,
    lr=5e-6,
    weight_decay=0.01,
    warmup_ratio=0.05,
    freeze_backbone_epochs=1
)

trainer2 = pl.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    precision="16-mixed",
    logger=wandb_logger_ast2,
    callbacks=[early_stop_stage2, checkpoint_stage2],
    gradient_clip_val=1.0,
    log_every_n_steps=10
)

trainer2.fit(model_stage2, datamodule=ast_data_module_stage2)

stage2_ckpt = checkpoint_stage2.best_model_path

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model           │ ASTForAudioClassification │ 86.2 M │ eval  │     0 │
│ 1 │ chunk_attention │ Sequential                │  1.5 K │ train │     0 │
│ 2 │ loss_fn         │ CrossEntropyLoss          │      0 │ train │     0 │
│ 3 │ val_f1          │ MulticlassF1Score         │      0 │ train │     0 │
└───┴─────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 86.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 86.2 M                                                                                               
Total estimated model params size (MB): 344                                                                        
Modules in train mode: 6                                                                                           
Modules in eval mode: 216                                                                                          
Total FLOPs: 0

Output()

## 3rd stage traning

In [86]:

wandb_logger_ast3 = WandbLogger(
    project="mashup-ast-final",
    log_model=False,
)

early_stop_stage3 = EarlyStopping(
    monitor="val_macro_f1",
    mode="max",
    patience=4,
    min_delta=0.001
)

checkpoint_stage3 = ModelCheckpoint(
    monitor="val_macro_f1",
    mode="max",
    save_top_k=1,
    filename="best-ast-stage3-{epoch}-{val_macro_f1:.4f}"
)

ast_data_module_stage3 = ASTMashupDataModule(
    train_dir="/kaggle/input/notebooks/a6du101/masups/mashups_adv",
    batch_size=12
)
model_stage3 = ASTLightning.load_from_checkpoint(
    stage2_ckpt,
    lr=3e-6,
    weight_decay=0.07,
    warmup_ratio=0.12,
    freeze_backbone_epochs=1
)
trainer3 = pl.Trainer(
    max_epochs=12,
    accelerator="auto",
    devices=1,
    precision="16-mixed",
    logger=wandb_logger_ast3,
    callbacks=[early_stop_stage3, checkpoint_stage3],
    gradient_clip_val=1.5,
    log_every_n_steps=10
)

trainer3.fit(model_stage3, datamodule=ast_data_module_stage3)

stage3_ckpt = checkpoint_stage3.best_model_path

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model           │ ASTForAudioClassification │ 86.2 M │ eval  │     0 │
│ 1 │ chunk_attention │ Sequential                │  1.5 K │ train │     0 │
│ 2 │ loss_fn         │ CrossEntropyLoss          │      0 │ train │     0 │
│ 3 │ val_f1          │ MulticlassF1Score         │      0 │ train │     0 │
└───┴─────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 86.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 86.2 M                                                                                               
Total estimated model params size (MB): 344                                                                        
Modules in train mode: 6                                                                                           
Modules in eval mode: 216                                                                                          
Total FLOPs: 0

Output()

# AST 3 stage Trained Model Kagglehub Upload

In [87]:
import kagglehub
model_path_ast = stage3_ckpt


KAGGLE_USERNAME = "a6du101"
FRAMEWORK = "pytorch"
VARIATION = "v1"
MODEL = "AST_final_s3"

handle_ast = f"{KAGGLE_USERNAME}/{MODEL}/{FRAMEWORK}/{VARIATION}"


kagglehub.model_upload(
    handle_ast,
    model_path_ast,
    version_notes="AST stage 3 "
)

print(f"Uploaded AST model to {handle_ast}")

Uploading Model https://api.kaggle.com/models/a6du101/AST_final_s3/pytorch/v1 ...
Model 'AST_final_s3' does not exist or access is forbidden for user 'a6du101'. Creating or handling Model...
Model 'AST_final_s3' Created.
Starting upload for file ./mashup-ast-final/k30y483o/checkpoints/best-ast-stage3-epoch=6-val_macro_f1=0.9461.ckpt


Uploading: 100%|██████████| 1.03G/1.03G [00:07<00:00, 138MB/s] 

Upload successful: ./mashup-ast-final/k30y483o/checkpoints/best-ast-stage3-epoch=6-val_macro_f1=0.9461.ckpt (987MB)


Your model instance has been created.
Files are being processed...
See at: https://api.kaggle.com/models/a6du101/AST_final_s3/pytorch/v1
Uploaded AST model to a6du101/AST_final_s3/pytorch/v1


# ResNET

## dataset

In [ ]:
class ResNetMashupDataset(Dataset):
    def __init__(self, root_dir, train=True, return_filenames=False):
        self.files = sorted(list(Path(root_dir).glob("*.wav")))
        self.train = train
        self.return_filenames = return_filenames

        self.sr = 22050
        self.target_duration = 10
        self.target_samples = self.sr * self.target_duration

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]

        y, _ = librosa.load(file, sr=self.sr)
        
        if np.abs(y).max() < 1e-5:
            print(f"LOW SIGNAL FILE: {file}")
        if len(y) > self.target_samples:
            if self.train:
                start = np.random.randint(0, len(y) - self.target_samples)
            else:
                start = (len(y) - self.target_samples) // 2
            y = y[start:start + self.target_samples]
        else:
            y = np.pad(y, (0, self.target_samples - len(y)))

        mel = librosa.feature.melspectrogram(
            y=y,
            sr=self.sr,
            n_fft=1024,
            hop_length=512,
            n_mels=128
        )

        mel = librosa.power_to_db(mel).astype(np.float32)
        mean = mel.mean()
        std = mel.std()
        
        if std < 1e-6:
            std = 1e-6
        
        mel = (mel - mean) / std

        mel = torch.tensor(mel).unsqueeze(0)  # (1, H, W)
        mel = mel.repeat(3, 1, 1)             # (3, H, W)

        if self.return_filenames:
            return mel, file.name

        genre = file.stem.split("_")[-1]
        y = torch.tensor(GENRE_TO_IDX[genre])

        return mel, y

## datamodule

In [ ]:
class ResNetDataModule(pl.LightningDataModule):
    def __init__(self, train_dir=None, test_dir=None, batch_size=16):
        super().__init__()
        self.train_dir = train_dir
        self.test_dir = test_dir
        self.batch_size = batch_size

    def setup(self, stage=None):
        if self.train_dir:
            full = ResNetMashupDataset(self.train_dir, train=True)

            val_size = int(0.2 * len(full))
            train_size = len(full) - val_size

            self.train_ds, self.val_ds = random_split(full, [train_size, val_size])
            self.val_ds.dataset.train = False

        if self.test_dir:
            self.test_ds = ResNetMashupDataset(
                self.test_dir,
                train=False,
                return_filenames=True
            )

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=4)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False, num_workers=4)

    def predict_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size, shuffle=False, num_workers=4)

In [ ]:
import timm

class ResNetLightning(pl.LightningModule):
    def __init__(self, n_classes=10, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()

        self.model = timm.create_model(
            "resnet18",
            pretrained=True,
            num_classes=n_classes
        )

        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

        self.val_f1 = torchmetrics.F1Score(
            task="multiclass",
            num_classes=n_classes,
            average="macro"
        )

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
    
        logits = self(x)
        loss = self.loss_fn(logits, y)
    
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)

        preds = torch.argmax(logits, dim=1)
        self.val_f1(preds, y)

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_macro_f1", self.val_f1, prog_bar=True)
        
    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        x, filenames = batch   # B C H W
    
        logits = self(x)
    
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(probs, dim=1)
    
        return {
            "preds": preds,
            "filenames": filenames
        }

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=10
        )
        return [optimizer], [scheduler]

## training

In [ ]:
wandb_logger_resnet = WandbLogger(
    project="Resnet18",
    log_model=False,
)

early_stop_res = EarlyStopping(
    monitor="val_macro_f1",
    mode="max",
    patience=4,
    min_delta=0.001
)

checkpoint_res = ModelCheckpoint(
    monitor="val_macro_f1",
    mode="max",
    save_top_k=1,
    filename="best-ResNet-{epoch}-{val_macro_f1:.4f}"
)

lr_monitor = LearningRateMonitor(logging_interval="epoch")

res_data_module = ResNetDataModule(
    train_dir="/kaggle/input/notebooks/a6du101/mashup-creation/mashups_v5",
    batch_size=16
)

model_res = ResNetLightning(
    n_classes=10,
    lr=3e-4
)

trainer_res = pl.Trainer(
    max_epochs=20,
    accelerator="auto",
    devices="auto",
    precision="16-mixed",
    logger=wandb_logger_resnet,
    callbacks=[early_stop_res, checkpoint_res, lr_monitor],
    gradient_clip_val=1.0,
    log_every_n_steps=10
)

# trainer_res.fit(model_res, res_data_module)

In [ ]:
# resnet_cpkt = checkpoint_res.best_model_path


# import kagglehub
# model_path = resnet_cpkt


# KAGGLE_USERNAME = "a6du101"
# FRAMEWORK = "pytorch"
# VARIATION = "v1"
# MODEL = "ResNET"

# handle = f"{KAGGLE_USERNAME}/{MODEL}/{FRAMEWORK}/{VARIATION}"


# kagglehub.model_upload(
#     handle,
#     model_path,
#     version_notes="Resnet traning with 10000 songs "
# )

# print(f"Uploaded AST model to {handle}")

# Inference AST

In [90]:
class MashupTestDatasetMultiCrop(Dataset):
    def __init__(self, root_dir):
        self.files = sorted(list(Path(root_dir).glob("*.wav")))
        self.feature_extractor = ASTFeatureExtractor.from_pretrained(
            "MIT/ast-finetuned-audioset-10-10-0.4593"
        )
        self.sr = 16000
        self.segment_samples = self.sr * 10

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]
        waveform, _ = librosa.load(file, sr=self.sr)

        crops = []

        for i in range(3):
            start = i * self.segment_samples
            end = start + self.segment_samples

            if end <= len(waveform):
                segment = waveform[start:end]
            else:
                segment = np.pad(
                    waveform[start:], 
                    (0, self.segment_samples - len(waveform[start:]))
                )

            inputs = self.feature_extractor(
                segment,
                sampling_rate=self.sr,
                return_tensors="pt"
            )

            crops.append(inputs["input_values"].squeeze(0))

        crops = torch.stack(crops)  # (3, seq_len, featuares_dim)

        return crops, file.name


class MashupInferenceDataset(Dataset):
    def __init__(self, root_dir):
        self.files = sorted(list(Path(root_dir).glob("*.wav")))
        self.feature_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
        self.target_sr = 16000
        self.win_len = 10  # 10 seconds per crop
        self.total_len = 30 # Full length of test audio

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]
        waveform, _ = librosa.load(file, sr=self.target_sr)
        
        target_samples = self.target_sr * self.total_len
        if len(waveform) < target_samples:
            waveform = np.pad(waveform, (0, target_samples - len(waveform)))
        else:
            waveform = waveform[:target_samples]

        # Crops: 0-10s, 5-15s, 10-20s, 15-25s, 20-30s
        crops = []
        stride = 5 * self.target_sr
        window = 10 * self.target_sr
        
        for start in range(0, target_samples - window + 1, stride):
            segment = waveform[start : start + window]
            
            inputs = self.feature_extractor(
                segment, 
                sampling_rate=self.target_sr, 
                return_tensors="pt"
            )
            crops.append(inputs["input_values"].squeeze(0))
        
        # Stack crops: Shape [5, 1024, 128]
        crops_tensor = torch.stack(crops)
        
        return crops_tensor, file.name


class ASTMashupInferenceDataset(Dataset):
    def __init__(self, root_dir, return_filenames=True):
        self.files = sorted(list(Path(root_dir).rglob("*.wav")))  # ✅ recursive
        self.return_filenames = return_filenames

        self.feature_extractor = ASTFeatureExtractor.from_pretrained(
            "MIT/ast-finetuned-audioset-10-10-0.4593"
        )

        self.target_sr = 16000
        self.target_duration = 30
        self.target_samples = self.target_sr * self.target_duration

        print(f"Found {len(self.files)} files for inference")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]

        waveform, sr = librosa.load(file, sr=self.target_sr)

        chunks = []
        total_len = len(waveform)

        # ✅ deterministic full coverage
        if total_len >= self.target_samples:
            n_chunks = total_len // self.target_samples

            for i in range(n_chunks):
                start = i * self.target_samples
                chunk = waveform[start:start + self.target_samples]
                chunks.append(chunk)
        else:
            padded = np.pad(waveform, (0, self.target_samples - total_len))
            chunks.append(padded)

        # ✅ feature extraction
        inputs = self.feature_extractor(
            chunks,
            sampling_rate=self.target_sr,
            return_tensors="pt"
        )

        x = inputs["input_values"]  # [N, T, F]

        if self.return_filenames:
            return x, file.name

        return x

# test_dataset = MashupTestDatasetMultiCrop(
#     "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"
# )

# test_loader = DataLoader(
#     test_dataset,
#     batch_size=4,
#     shuffle=False,
#     num_workers=3,
# )

# test_dataset = MashupInferenceDataset(
#     "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"
# )

# test_loader = DataLoader(
#     test_dataset,
#     batch_size=8,
#     shuffle=False,
#     num_workers=4,
# )
test_dataset = ASTMashupInferenceDataset(
    "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"
)
test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=4,
)

# # average
# stage2_ckpt_path = "/kaggle/input/models/a6du101/ast-step1-trained/pytorch/v1/2/best-ast-stage2-epoch5-val_macro_f10.9765.ckpt"
# stage3_ckpt_path = "/kaggle/input/models/a6du101/ast-step3-trained/pytorch/v1/1/best-ast-stage3-epoch5-val_macro_f10.9264.ckpt"

stage2_ckpt_path = stage2_ckpt
stage3_ckpt_path = stage3_ckpt

ckpt2 = torch.load(stage2_ckpt_path, map_location="cpu")
ckpt3 = torch.load(stage3_ckpt_path, map_location="cpu")

state2 = ckpt2["state_dict"]
state3 = ckpt3["state_dict"]

print(len(state2), len(state3))

for k in state2:
    if k not in state3:
        print("Missing key:", k)


alpha = 0.5  # weight for stage2
beta = 0.5   # weight for stage3

avg_state = {}

for k in state2.keys():
    avg_state[k] = alpha * state2[k] + beta * state3[k]



avg_ckpt = ckpt3.copy()
avg_ckpt["state_dict"] = avg_state

avg_ckpt_path = "/kaggle/working/ast_stage2_stage3_avg.ckpt"
torch.save(avg_ckpt, avg_ckpt_path)

print("Saved averaged checkpoint to:", avg_ckpt_path)

# ckpt_path = "/kaggle/input/models/a6du101/mashup-ast-3stage-2/pytorch/v1/1/best-ast-stage3-epoch3-val_macro_f10.9131.ckpt"

model = ASTLightning.load_from_checkpoint(
    checkpoint_path=avg_ckpt_path,
    n_classes=10
)

Found 3020 files for inference
207 207
Saved averaged checkpoint to: /kaggle/working/ast_stage2_stage3_avg.ckpt


Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# class ResNetInferDataset(Dataset):
#     def __init__(self, root_dir):
#         self.files = sorted(list(Path(root_dir).glob("*.wav")))
#         self.sr = 22050
#         self.target_samples = self.sr * 30  # full 30s

#     def __len__(self):
#         return len(self.files)

#     def __getitem__(self, idx):
#         file = self.files[idx]

#         y, _ = librosa.load(file, sr=self.sr)

#         # pad / trim to 30s
#         if len(y) < self.target_samples:
#             y = np.pad(y, (0, self.target_samples - len(y)))
#         else:
#             y = y[:self.target_samples]

#         mel = librosa.feature.melspectrogram(
#             y=y,
#             sr=self.sr,
#             n_fft=1024,
#             hop_length=512,
#             n_mels=128
#         )

#         mel = librosa.power_to_db(mel).astype(np.float32)

#         # safe normalization
#         mean = mel.mean()
#         std = mel.std()
#         if std < 1e-6:
#             std = 1e-6

#         mel = (mel - mean) / std
#         mel = np.clip(mel, -5, 5)

#         mel = torch.tensor(mel).unsqueeze(0).repeat(3, 1, 1)

#         return mel, file.name


# test_dataset = ResNetInferDataset(
#     "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"
# )

# test_loader = DataLoader(
#     test_dataset,
#     batch_size=4,
#     shuffle=False,
#     num_workers=3,
# )

# model = ResNetLightning.load_from_checkpoint(
#     "/kaggle/input/models/a6du101/resnet-30s/pytorch/v1/1/best-ResNet-epoch9-val_macro_f10.9773.ckpt"   # change path
# )

In [91]:
trainer = pl.Trainer(accelerator="auto", devices="auto")

predictions = trainer.predict(model, dataloaders=test_loader)

all_preds = []
all_filenames = []

for batch in predictions:
    all_preds.extend(batch["preds"].cpu().numpy())
    all_filenames.extend(batch["filenames"])

predicted_genres = [IDX_TO_GENRE[p] for p in all_preds]

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Output()

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e176292d080><function _MultiProcessingDataLoaderIter.__del__ at 0x7e176292d080>

Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():    
if w.is_alive(): 
  Exception ignored in:   <function _MultiProcessingDataLoaderIter.__del__ at 0x7e176292d080>  
 Exception ignored in: Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/d

In [ ]:
# model = model.cuda()
# model.eval()

# all_preds = []
# all_filenames = []

# with torch.no_grad():
#     for crops, filenames in test_loader:

#         B, N, T, F = crops.shape

#         crops = crops.view(B * N, T, F).cuda()

#         logits = model(crops)  # (B*N, num_classes)

#         probs = torch.softmax(logits, dim=-1)
#         probs = probs.view(B, N, -1)
#         probs = probs.mean(dim=1)

#         preds = torch.argmax(probs, dim=1)

#         all_preds.extend(preds.cpu().numpy())
#         all_filenames.extend(filenames)

# predicted_genres = [IDX_TO_GENRE[p] for p in all_preds]

In [ ]:
# predicted_genres = [IDX_TO_GENRE[p] for p in all_preds]

In [92]:
len(predicted_genres)

3020

In [93]:
test_df = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv")

print(len(predicted_genres), len(test_df))

test_df["genre"] = predicted_genres

submission = test_df[["id", "genre"]]
submission.to_csv("submission.csv", index=False)

submission.head()

3020 3020


,id,genre
0,1,pop
1,2,classical
2,3,disco
3,4,metal
4,5,country


In [94]:
submission.genre.value_counts(normalize=False)

genre
rock         394
pop          336
hiphop       333
jazz         317
reggae       316
disco        312
metal        309
blues        293
classical    206
country      204
Name: count, dtype: int64

In [ ]:
# SR = 22050
# DURATION = 30  # seconds
# MAX_LEN = SR * DURATION

# def load_and_preprocess(path):
#     y, sr = librosa.load(path, sr=SR, mono=True)
    
#     # Normalize
#     y = librosa.util.normalize(y)
    
#     # Pad or trim
#     if len(y) > MAX_LEN:
#         y = y[:MAX_LEN]
#     else:
#         y = np.pad(y, (0, MAX_LEN - len(y)))
        
#     return y


# def extract_features(y):
#     # MFCC
#     mfcc = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=20)
#     mfcc_delta = librosa.feature.delta(mfcc)
    
#     # Log-mel
#     mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=64)
#     log_mel = librosa.power_to_db(mel)
    
#     features = np.hstack([
#         mfcc.mean(axis=1),
#         mfcc.std(axis=1),
#         mfcc_delta.mean(axis=1),
#         mfcc_delta.std(axis=1),
#         log_mel.mean(axis=1),
#         log_mel.std(axis=1)
#     ])
    
#     return features

In [ ]:
# from joblib import Parallel, delayed
# import os

# stems = ["vocals.wav", "drums.wav", "bass.wav", "other.wav"]

# def process_song(genre, song):
#     song_path = os.path.join(training_path, genre, song)
#     stem_features = []

#     for stem in stems:
#         path = os.path.join(song_path, stem)
#         audio = load_and_preprocess(path)
#         feats = extract_features(audio)
#         stem_features.append(feats)
#     return np.hstack(stem_features), genre


# # Build task list
# tasks = []
# for genre in os.listdir(training_path):
#     for song in os.listdir(os.path.join(training_path, genre)):
#         tasks.append((genre, song))

# # Parallel with tqdm
# results = Parallel(n_jobs=-1)(
#     delayed(process_song)(g, s) 
#     for g, s in tqdm(tasks, total=len(tasks))
# )

# X = np.array([r[0] for r in results])
# y = np.array([r[1] for r in results])

In [ ]:


# le = LabelEncoder()
# y_le = le.fit_transform(y)

In [ ]:


# X_train, X_val, y_train, y_val = train_test_split(
#     X, y_le,
#     test_size=0.2,
#     stratify=y_le,
#     random_state=42
# )

In [ ]:
# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_val = scaler.transform(X_val)

In [ ]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import f1_score

# model = LogisticRegression(max_iter=2000)
# model.fit(X_train, y_train)

# y_pred_log = model.predict(X_val)

# macro_f1 = f1_score(y_val, y_pred, average="macro")
# print("Logistic Regression Macro F1:", macro_f1)

In [ ]:
# from sklearn.naive_bayes import GaussianNB

# nb = GaussianNB()
# nb.fit(X_train, y_train)

# y_pred_nb = nb.predict(X_val)
# nb_f1 =f1_score(y_val, y_pred, average="macro")
# print("Naive Bayes Macro F1:",nb_f1)

In [ ]:
# from xgboost import XGBClassifier

# xgb = XGBClassifier(
#     n_estimators=300,
#     max_depth=6,
#     learning_rate=0.05,
#     objective="multi:softmax",
#     num_class=len(le.classes_),
#     eval_metric="mlogloss"
# )

# xgb.fit(X_train, y_train)

# y_pred_xgb = xgb.predict(X_val)
# print("XGBoost Macro F1:", f1_score(y_val, y_pred, average="macro"))

In [ ]:
# xgb_f1= f1_score(y_val, y_pred, average="macro")
# xgb_f1

In [ ]:
# from catboost import CatBoostClassifier
# from sklearn.metrics import f1_score

# cat = CatBoostClassifier(
#     iterations=500,
#     depth=6,
#     learning_rate=0.05,
#     loss_function="MultiClass",
#     eval_metric="TotalF1",
#     random_seed=42,
#     verbose=100
# )

# cat.fit(X_train, y_train, eval_set=(X_val, y_val))

# y_pred_cat = cat.predict(X_val)
# y_pred_cat = y_pred_cat.astype(int)

# cat_f1 = f1_score(y_val, y_pred_cat, average="macro")
# print("CatBoost Macro F1:", cat_f1)

In [ ]:
# from lightgbm import LGBMClassifier

# lgb_model = LGBMClassifier(
#     n_estimators=500,
#     learning_rate=0.05,
#     max_depth=-1,
#     num_leaves=31,
#     objective="multiclass",
#     num_class=len(le.classes_),
#     random_state=42
# )

# lgb_model.fit(X_train, y_train)

# y_pred_lgb = lgb_model.predict(X_val)

# lgb_f1 = f1_score(y_val, y_pred_lgb, average="macro")
# print("LightGBM Macro F1:", lgb_f1)

In [ ]:
# gbc=GradientBoostingClassifier(random_state=42)
# gbc.fit(X_train, y_train)

# y_pred_gbc = gbc.predict(X_val)

# gbc_f1 = f1_score(y_val, y_pred_gbc, average="macro")
# print("GBClassifier Macro F1:", gbc_f1)

In [ ]:

# wandb.init(project="classical-ml-baseline", name="gbclassifier")

# wandb.log({
#     'gbclassifier_macro_f1': gbc_f1
# })
# wandb.log({
#     "logreg_macro_f1": macro_f1,
#     "catboost_macro_f1": cat_f1,
#     "lightgbm_macro_f1": lgb_f1,
#     "naive_bayes_macro_f1": nb_f1,
#     "xgb_macro_f1":xgb_f1
# })

In [ ]:

# wandb.finish()